In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# ============================================================
# 2. Install dan Load YOLOv8s
# ============================================================
!pip install -U ultralytics -q

from ultralytics import YOLO

model = YOLO("yolov8s.pt")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# ============================================================
# 3. Unzip Dataset Helmet Detection
# ============================================================
import os

!rm -rf /content/dataset
!mkdir -p /content/dataset

# GANTI SESUAI NAMA FILE ZIP DATASET KAMU DI GOOGLE DRIVE
DATASET_ZIP = "/content/drive/MyDrive/helmet detection.v2i.yolov8.zip"

!unzip -q "{DATASET_ZIP}" -d /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [ ]:
# ============================================================
# 4. Cari data.yaml Otomatis
# ============================================================
import os

dataset_path = None

for root, dirs, files in os.walk("/content/dataset"):
    if "data.yaml" in files:
        dataset_path = root
        break

if dataset_path is None:
    raise FileNotFoundError("data.yaml tidak ditemukan. Cek lagi file zip dataset.")

print("Dataset path:", dataset_path)
print("Isi folder dataset:", os.listdir(dataset_path))

Dataset path: /content/dataset
Isi folder dataset: ['data.yaml', 'README.dataset.txt', 'valid', 'train', 'README.roboflow.txt', 'test']


In [ ]:
# ============================================================
# 5. Baca data.yaml
# ============================================================
import yaml
import os

yaml_path = os.path.join(dataset_path, "data.yaml")

with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print("Data YAML awal:")
print(data_yaml)

Data YAML awal:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 6. Fix Path data.yaml untuk Colab
# ============================================================
import os
import yaml

train_dir = os.path.join(dataset_path, "train/images")
valid_dir = os.path.join(dataset_path, "valid/images")
val_dir   = os.path.join(dataset_path, "val/images")
test_dir  = os.path.join(dataset_path, "test/images")

data_yaml["train"] = train_dir

if os.path.exists(valid_dir):
    data_yaml["val"] = valid_dir
elif os.path.exists(val_dir):
    data_yaml["val"] = val_dir
else:
    raise FileNotFoundError("Folder valid/images atau val/images tidak ditemukan.")

if os.path.exists(test_dir):
    data_yaml["test"] = test_dir
else:
    print("Peringatan: folder test/images tidak ditemukan. Evaluasi test nanti bisa gagal.")

# Rapikan format names
if "names" in data_yaml:
    names = data_yaml["names"]

    if isinstance(names, dict):
        # Aman untuk key string atau integer
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]

    data_yaml["names"] = names
    data_yaml["nc"] = len(names)

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Data YAML setelah diperbaiki:")
print(data_yaml)

Data YAML setelah diperbaiki:
{'train': '/content/dataset/train/images', 'val': '/content/dataset/valid/images', 'test': '/content/dataset/test/images', 'nc': 3, 'names': ['helmet', 'no_helmet', 'person'], 'roboflow': {'workspace': '039s-workspace', 'project': 'helmet-detection-wagvw', 'version': 2, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/039s-workspace/helmet-detection-wagvw/dataset/2'}}


In [ ]:
# ============================================================
# 7. Cek Jumlah Dataset
# ============================================================
from pathlib import Path

def get_split_folder(split):
    if split == "valid":
        if (Path(dataset_path) / "valid").exists():
            return "valid"
        elif (Path(dataset_path) / "val").exists():
            return "val"
    return split

def count_dataset(split):
    split_folder = get_split_folder(split)

    img_dir = Path(dataset_path) / split_folder / "images"
    label_dir = Path(dataset_path) / split_folder / "labels"

    images = list(img_dir.glob("*")) if img_dir.exists() else []
    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

    print(f"{split_folder}")
    print("Images :", len(images))
    print("Labels :", len(labels))
    print("-" * 30)

for split in ["train", "valid", "test"]:
    count_dataset(split)

train
Images : 6482
Labels : 6482
------------------------------
valid
Images : 1671
Labels : 1671
------------------------------
test
Images : 869
Labels : 869
------------------------------


In [ ]:
# ============================================================
# 8. Cleaning Dataset
# Hapus label kosong, gambar tanpa label, dan label tanpa gambar
# ============================================================
from pathlib import Path

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

def clean_dataset(split):
    split_folder = get_split_folder(split)

    img_dir = Path(dataset_path) / split_folder / "images"
    label_dir = Path(dataset_path) / split_folder / "labels"

    if not img_dir.exists() or not label_dir.exists():
        print(f"Split {split_folder} tidak ditemukan, dilewati.")
        return

    print(f"\nMembersihkan split: {split_folder}")

    # Hapus label kosong
    empty_count = 0
    for label_path in label_dir.glob("*.txt"):
        if label_path.stat().st_size == 0:
            label_path.unlink()
            empty_count += 1

    # Hapus gambar tanpa label
    removed_images = 0
    for img_path in img_dir.iterdir():
        if img_path.suffix in IMG_EXTS:
            label_path = label_dir / (img_path.stem + ".txt")
            if not label_path.exists():
                img_path.unlink()
                removed_images += 1

    # Hapus label tanpa gambar
    image_stems = {p.stem for p in img_dir.iterdir() if p.suffix in IMG_EXTS}
    removed_labels = 0

    for label_path in label_dir.glob("*.txt"):
        if label_path.stem not in image_stems:
            label_path.unlink()
            removed_labels += 1

    print("Label kosong dihapus:", empty_count)
    print("Gambar tanpa label dihapus:", removed_images)
    print("Label tanpa gambar dihapus:", removed_labels)

for split in ["train", "valid", "test"]:
    clean_dataset(split)


Membersihkan split: train
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: valid
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0

Membersihkan split: test
Label kosong dihapus: 0
Gambar tanpa label dihapus: 0
Label tanpa gambar dihapus: 0


In [ ]:
# ============================================================
# 9. Validasi Format Label YOLO
# Format benar: class x_center y_center width height
# ============================================================
from pathlib import Path

NC = data_yaml["nc"]
print("Jumlah kelas:", NC)
print("Nama kelas:", data_yaml["names"])

def validate_labels(split):
    split_folder = get_split_folder(split)
    label_dir = Path(dataset_path) / split_folder / "labels"

    if not label_dir.exists():
        print(f"{split_folder} labels tidak ditemukan, dilewati.")
        return []

    bad_files = []

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):
            parts = line.strip().split()

            if len(parts) != 5:
                bad_files.append((label_path.name, line_number, "jumlah kolom bukan 5", line.strip()))
                break

            try:
                cls = int(float(parts[0]))
                values = list(map(float, parts[1:]))
            except:
                bad_files.append((label_path.name, line_number, "format angka salah", line.strip()))
                break

            if cls < 0 or cls >= NC:
                bad_files.append((label_path.name, line_number, f"class id salah: {cls}", line.strip()))
                break

            if any(v < 0 or v > 1 for v in values):
                bad_files.append((label_path.name, line_number, "koordinat di luar 0-1", line.strip()))
                break

    print(f"{split_folder} label bermasalah:", len(bad_files))

    for item in bad_files[:10]:
        print(item)

    return bad_files

all_bad = []

for split in ["train", "valid", "test"]:
    all_bad.extend(validate_labels(split))

print("Total semua label bermasalah:", len(all_bad))

Jumlah kelas: 3
Nama kelas: ['helmet', 'no_helmet', 'person']
train label bermasalah: 0
valid label bermasalah: 0
test label bermasalah: 1
('005377_jpg.rf.877294944cd6d730dd900feb00280ffe.txt', 2, 'jumlah kolom bukan 5', '0 0.5925 0.975 0.5900000000000001 0.9016666666666666 0.5775 0.8783333333333333 0.5575 0.7183333333333334 0.60375 0.7 0.6375 0.8816666666666666 0.675 0.8716666666666667 0.6625 0.7016666666666667 0.665 0.6716666666666666 0.7025 0.5916666666666667 0.7125 0.3983333333333333 0.7025 0.335 0.6625 0.2783333333333333 0.65 0.23166666666666663 0.6525000000000001 0.19833333333333333 0.6162500000000001 0.16 0.58875 0.16666666666666669 0.5650000000000001 0.19166666666666668 0.5625 0.24833333333333335 0.5650000000000001 0.2783333333333333 0.5825 0.29166666666666663 0.60375 0.33666666666666667 0.6225 0.3416666666666667 0.62 0.3783333333333333 0.60875 0.4066666666666666 0.5537500000000001 0.4 0.535 0.4116666666666666 0.5375 0.43166666666666664 0.5925 0.505 0.5900000000000001 0.5916666

In [ ]:
# ============================================================
# 10. Training YOLOv8s Helmet Detection
# ============================================================
import time
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

start_train = time.time()

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/helmet_detection_yolo",
    name="yolov8s_helmet_detection_img640",
    workers=2,
    patience=30,
    save=True,
    plots=True,
    cache=False,
    cos_lr=True,
    close_mosaic=10
)

end_train = time.time()

training_time_seconds = end_train - start_train
training_time_minutes = training_time_seconds / 60
training_time_hours = training_time_seconds / 3600

print("Training time:", training_time_seconds, "detik")
print("Training time:", training_time_minutes, "menit")
print("Training time:", training_time_hours, "jam")

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_helmet_detection_img640, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [ ]:
# ============================================================
# 11. Evaluasi Best Model YOLOv8s pada Test Set
# ============================================================
from ultralytics import YOLO
import os

best_model_path = "/content/helmet_detection_yolo/yolov8s_helmet_detection_img640/weights/best.pt"

if not os.path.exists(best_model_path):
    raise FileNotFoundError("best.pt tidak ditemukan. Pastikan training YOLOv8s sudah selesai.")

model = YOLO(best_model_path)

metrics = model.val(
    data=yaml_path,
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project="/content/helmet_detection_eval",
    name="final_test_yolov8s_img640",
    exist_ok=True
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
f1_score = 2 * (precision * recall) / (precision + recall + 1e-9)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

print("Precision :", precision)
print("Recall    :", recall)
print("F1-score  :", f1_score)
print("mAP@50    :", map50)
print("mAP@50:95 :", map50_95)

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 19.0±14.3 MB/s, size: 60.7 KB)
val: Scanning /content/dataset/test/labels... 869 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 869/869 511.1it/s 1.7s
val: New cache created: /content/dataset/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 3, len(boxes) = 3271. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 7.6it/s 14.3s
                   all        869       3271      0.828      0.802      0.839      0.558
                helmet        716       2110      0.914      0.901      0.943   

In [ ]:
# ============================================================
# 11.1 Simpan Per-Class mAP@50:95 ke CSV
# ============================================================
import pandas as pd
import os

OUTPUT_DIR = "/content/helmet_detection_eval/yolov8s_final_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class_names = data_yaml["names"]

per_class_rows = []

# metrics.box.maps biasanya berisi mAP@50:95 per class
maps_per_class = getattr(metrics.box, "maps", [])

for i, class_name in enumerate(class_names):
    value = float(maps_per_class[i]) if i < len(maps_per_class) else None
    per_class_rows.append({
        "Model": "YOLOv8s",
        "Class": class_name,
        "mAP@50:95": value
    })

per_class_df = pd.DataFrame(per_class_rows)

per_class_path = os.path.join(OUTPUT_DIR, "yolov8s_per_class_metrics.csv")
per_class_df.to_csv(per_class_path, index=False)

per_class_df

,Model,Class,mAP@50:95
0,YOLOv8s,helmet,0.623944
1,YOLOv8s,no_helmet,0.654206
2,YOLOv8s,person,0.395042


In [ ]:
# ============================================================
# 12. Hitung Inference Time dan FPS YOLOv8s
# ============================================================
import glob
import os
import time
import torch

TEST_IMG_DIR = os.path.join(dataset_path, "test/images")

test_images = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_images.extend(glob.glob(os.path.join(TEST_IMG_DIR, ext)))

print("Jumlah gambar test:", len(test_images))

# Warm up
for img_path in test_images[:10]:
    _ = model.predict(img_path, imgsz=640, conf=0.25, verbose=False)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start_time = time.time()

for img_path in test_images:
    _ = model.predict(img_path, imgsz=640, conf=0.25, verbose=False)

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.time()

total_inference_time = end_time - start_time
avg_inference_time = total_inference_time / len(test_images)
fps = 1 / avg_inference_time

print("Total inference time:", total_inference_time, "detik")
print("Average inference time:", avg_inference_time, "detik/gambar")
print("FPS:", fps)

Jumlah gambar test: 869
Total inference time: 12.065048694610596 detik
Average inference time: 0.013883830488619789 detik/gambar
FPS: 72.02623230092544


In [ ]:
# ============================================================
# 13. Simpan Metrik YOLOv8s ke CSV
# ============================================================
import pandas as pd
import os

OUTPUT_DIR = "/content/helmet_detection_eval/yolov8s_final_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

summary = pd.DataFrame([{
    "Model": "YOLOv8s",
    "Dataset": "Helmet Detection",
    "Jumlah Kelas": data_yaml["nc"],
    "Kelas": ", ".join(data_yaml["names"]),
    "Epochs": 100,
    "Image Size": 640,
    "Batch": 16,
    "Training Time (s)": training_time_seconds,
    "Training Time (minutes)": training_time_minutes,
    "Training Time (hours)": training_time_hours,
    "Accuracy": "",
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1_score,
    "mAP@50": map50,
    "mAP@50:95": map50_95,
    "Inference Time per Image (s)": avg_inference_time,
    "FPS": fps
}])

summary_path = os.path.join(OUTPUT_DIR, "yolov8s_final_metrics.csv")
summary.to_csv(summary_path, index=False)

summary

,Model,Dataset,Jumlah Kelas,Kelas,Epochs,Image Size,Batch,Training Time (s),Training Time (minutes),Training Time (hours),Accuracy,Precision,Recall,F1-Score,mAP@50,mAP@50:95,Inference Time per Image (s),FPS
0,YOLOv8s,Helmet Detection,3,"helmet, no_helmet, person",100,640,16,13750.129165,229.168819,3.81948,,0.828345,0.802125,0.815024,0.839277,0.557731,0.013884,72.026232


In [ ]:
# ============================================================
# 14. Simpan Sample Prediction YOLOv8s
# ============================================================
model.predict(
    source=TEST_IMG_DIR,
    imgsz=640,
    conf=0.25,
    save=True,
    project="/content/helmet_detection_eval",
    name="yolov8s_prediction_sample",
    exist_ok=True
)

Results saved to /content/helmet_detection_eval/yolov8s_prediction_sample


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'helmet', 1: 'no_helmet', 2: 'person'}
 obb: None
 orig_img: array([[[ 98, 117,  98],
         [102, 121, 102],
         [105, 125, 106],
         ...,
         [ 76,  94,  87],
         [ 74,  92,  85],
         [ 73,  91,  84]],
 
        [[ 99, 118,  99],
         [103, 122, 103],
         [106, 126, 107],
         ...,
         [ 76,  94,  87],
         [ 74,  92,  85],
         [ 73,  91,  84]],
 
        [[101, 120, 101],
         [105, 124, 105],
         [107, 127, 108],
         ...,
         [ 75,  93,  86],
         [ 73,  91,  84],
         [ 73,  91,  84]],
 
        ...,
 
        [[220, 236, 212],
         [221, 237, 213],
         [222, 238, 214],
         ...,
         [ 95, 129, 129],
         [ 83, 117, 117],
         [ 87, 121, 121]],
 
        [[220, 236, 212],
         [221, 237, 213],
         [222, 238, 214],
   

In [ ]:
# ============================================================
# 15. Kumpulkan Semua Hasil YOLOv8s ke Folder Final
# ============================================================
import os
import shutil
import glob

RESULT_FOLDER = "/content/YOLOv8s_Helmet_Detection_Final_Result"

if os.path.exists(RESULT_FOLDER):
    shutil.rmtree(RESULT_FOLDER)

os.makedirs(RESULT_FOLDER, exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/01_Model", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/02_Training_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/03_Evaluation_Result", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/04_Visualization", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/05_Prediction_Sample", exist_ok=True)
os.makedirs(f"{RESULT_FOLDER}/06_Config", exist_ok=True)

TRAIN_DIR = "/content/helmet_detection_yolo/yolov8s_helmet_detection_img640"
EVAL_DIR = "/content/helmet_detection_eval/final_test_yolov8s_img640"
PRED_DIR = "/content/helmet_detection_eval/yolov8s_prediction_sample"

# Copy model
if os.path.exists(f"{TRAIN_DIR}/weights/best.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/best.pt", f"{RESULT_FOLDER}/01_Model/best_yolov8s.pt")

if os.path.exists(f"{TRAIN_DIR}/weights/last.pt"):
    shutil.copy(f"{TRAIN_DIR}/weights/last.pt", f"{RESULT_FOLDER}/01_Model/last_yolov8s.pt")

# Copy hasil training
for file in glob.glob(f"{TRAIN_DIR}/*"):
    if os.path.isfile(file):
        shutil.copy(file, f"{RESULT_FOLDER}/02_Training_Result/{os.path.basename(file)}")

# Copy hasil evaluasi CSV
if os.path.exists(OUTPUT_DIR):
    for file in glob.glob(f"{OUTPUT_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy hasil evaluasi test
if os.path.exists(EVAL_DIR):
    for file in glob.glob(f"{EVAL_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/03_Evaluation_Result/{os.path.basename(file)}")

# Copy visualisasi training
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    for file in glob.glob(f"{TRAIN_DIR}/{ext}"):
        shutil.copy(file, f"{RESULT_FOLDER}/04_Visualization/{os.path.basename(file)}")

# Copy prediction sample
if os.path.exists(PRED_DIR):
    for file in glob.glob(f"{PRED_DIR}/*"):
        if os.path.isfile(file):
            shutil.copy(file, f"{RESULT_FOLDER}/05_Prediction_Sample/{os.path.basename(file)}")

# Copy config
shutil.copy(yaml_path, f"{RESULT_FOLDER}/06_Config/data.yaml")

# README
readme = f"""
YOLOv8s Helmet Detection Final Result

Judul:
Deteksi Kepatuhan Penggunaan Helm Keselamatan Menggunakan Perbandingan YOLOv5, YOLOv8, dan YOLOv11 pada Dataset Publik

Model:
YOLOv8s

Dataset:
Helmet Detection

Class:
{", ".join(data_yaml["names"])}

Setting Training:
- Epochs: 100
- Image Size: 640
- Batch: 16
- Optimizer: default Ultralytics
- Patience: 30
- cos_lr: True
- close_mosaic: 10
- cache: False
- Framework: Ultralytics YOLOv8

Metrik:
- Precision
- Recall
- F1-score
- mAP@50
- mAP@50:95
- Inference Time
- FPS
- Confusion Matrix
- Per-Class mAP
"""

with open(f"{RESULT_FOLDER}/README.txt", "w") as f:
    f.write(readme)

print("Folder hasil dibuat:", RESULT_FOLDER)

Folder hasil dibuat: /content/YOLOv8s_Helmet_Detection_Final_Result


In [ ]:
# ============================================================
# 16. Simpan Folder Final ke Google Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_TARGET = "/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv8s_Final_Result"

if os.path.exists(DRIVE_TARGET):
    shutil.rmtree(DRIVE_TARGET)

shutil.copytree(RESULT_FOLDER, DRIVE_TARGET)

print("Hasil YOLOv8s Helmet Detection sudah masuk ke Google Drive:")
print(DRIVE_TARGET)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hasil YOLOv8s Helmet Detection sudah masuk ke Google Drive:
/content/drive/MyDrive/UAS_AI_Helmet_Detection/YOLOv8s_Final_Result


In [ ]:
# ============================================================
# 17. Buat ZIP dan Download
# ============================================================
import shutil
from google.colab import files

shutil.make_archive("/content/YOLOv8s_Helmet_Detection_Final_Result", "zip", RESULT_FOLDER)

print("ZIP dibuat di: /content/YOLOv8s_Helmet_Detection_Final_Result.zip")

files.download("/content/YOLOv8s_Helmet_Detection_Final_Result.zip")

In [ ]:
# ============================================================
# Test Accuracy YOLOv8s
# Correct Detection / Total Ground Truth dengan IoU >= 0.5
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import os
import glob
import pandas as pd
import torch

# Path best model YOLOv8s
best_model_path = "/content/helmet_detection_yolo/yolov8s_helmet_detection_img640/weights/best.pt"

# Path test images dan labels
TEST_IMG_DIR = os.path.join(dataset_path, "test/images")
TEST_LABEL_DIR = os.path.join(dataset_path, "test/labels")

model = YOLO(best_model_path)

IMG_EXTS = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]

test_images = []
for ext in IMG_EXTS:
    test_images.extend(glob.glob(os.path.join(TEST_IMG_DIR, ext)))

print("Jumlah gambar test:", len(test_images))


def xywh_to_xyxy(box):
    x, y, w, h = box
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return [x1, y1, x2, y2]


def bbox_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter_area = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - inter_area

    if union == 0:
        return 0

    return inter_area / union


total_gt = 0
correct_detection = 0

per_class_total = {}
per_class_correct = {}

IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.25

for img_path in test_images:
    img_stem = Path(img_path).stem
    label_path = os.path.join(TEST_LABEL_DIR, img_stem + ".txt")

    if not os.path.exists(label_path):
        continue

    # Baca ground truth
    gt_boxes = []
    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls = int(float(parts[0]))
            box = list(map(float, parts[1:]))
            gt_boxes.append({
                "cls": cls,
                "box": xywh_to_xyxy(box)
            })

            total_gt += 1
            per_class_total[cls] = per_class_total.get(cls, 0) + 1

    # Prediksi model
    results = model.predict(
        img_path,
        imgsz=640,
        conf=CONF_THRESHOLD,
        verbose=False
    )

    pred_boxes = []

    for r in results:
        if r.boxes is None:
            continue

        boxes_xywhn = r.boxes.xywhn.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy()

        for box, cls in zip(boxes_xywhn, classes):
            pred_boxes.append({
                "cls": int(cls),
                "box": xywh_to_xyxy(box)
            })

    matched_pred = set()

    # Matching GT dengan prediksi berdasarkan class dan IoU
    for gt in gt_boxes:
        best_iou = 0
        best_pred_idx = -1

        for idx, pred in enumerate(pred_boxes):
            if idx in matched_pred:
                continue

            if pred["cls"] != gt["cls"]:
                continue

            iou = bbox_iou(gt["box"], pred["box"])

            if iou > best_iou:
                best_iou = iou
                best_pred_idx = idx

        if best_iou >= IOU_THRESHOLD:
            correct_detection += 1
            matched_pred.add(best_pred_idx)

            cls = gt["cls"]
            per_class_correct[cls] = per_class_correct.get(cls, 0) + 1


accuracy = correct_detection / total_gt if total_gt > 0 else 0

print("===================================")
print("YOLOv8s Test Accuracy")
print("===================================")
print("Total Ground Truth :", total_gt)
print("Correct Detection  :", correct_detection)
print("Accuracy           :", accuracy)
print("Accuracy (%)       :", accuracy * 100)
print("===================================")

# Per-class accuracy
class_names = data_yaml["names"]

rows = []

for cls_id, total in per_class_total.items():
    correct = per_class_correct.get(cls_id, 0)
    acc = correct / total if total > 0 else 0

    rows.append({
        "Class ID": cls_id,
        "Class": class_names[cls_id],
        "Total GT": total,
        "Correct Detection": correct,
        "Accuracy": acc,
        "Accuracy (%)": acc * 100
    })

df_accuracy = pd.DataFrame(rows)
df_accuracy

Jumlah gambar test: 869
YOLOv8s Test Accuracy
Total Ground Truth : 3269
Correct Detection  : 2970
Accuracy           : 0.9085347200978893
Accuracy (%)       : 90.85347200978893


,Class ID,Class,Total GT,Correct Detection,Accuracy,Accuracy (%)
0,0,helmet,2108,1965,0.932163,93.216319
1,2,person,418,312,0.746411,74.641148
2,1,no_helmet,743,693,0.932705,93.270525


In [ ]:
OUTPUT_DIR = "/content/helmet_detection_eval/yolov8s_final_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

accuracy_path = os.path.join(OUTPUT_DIR, "yolov8s_test_accuracy.csv")
df_accuracy.to_csv(accuracy_path, index=False)

print("CSV disimpan di:", accuracy_path)

CSV disimpan di: /content/helmet_detection_eval/yolov8s_final_evaluation/yolov8s_test_accuracy.csv
